In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from pathlib import Path
import pandas as pd
import ast
from PIL import Image, ImageDraw, ImageFont
import math

In [8]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common room',
    (0xFF, 0xA5, 0x00): 'master room',
    (0xEE, 0xE8, 0xAA): 'living room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

ANNOT_DIR = Path("../../annotation/human_annotated_tags")

In [ ]:
# class LocalVisionLLM:
#     def __init__(self, model_id: str, device: str = 'cuda'):
#         # Processor for both vision & text
#         self.processor = AutoProcessor.from_pretrained(
#             model_id,
#             use_auth_token=True,
#             trust_remote_code=True
#         )

#         # Vision‑language conditional generation model
#         self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#             model_id,
#             torch_dtype=torch.float16,
#             device_map='auto',
#             trust_remote_code=True,
#             use_auth_token=True
#         )
#         # track actual device (could be spread across GPUs)
#         self.device = next(self.model.parameters()).device

#     def __call__(self, images: list[Image.Image], prompt: str, **generate_kwargs) -> str:
#         # 1) Build the “chat” messages list
#         messages = [{"type": "text",  "content": prompt}]
#         messages += [{"type": "image", "content": img} for img in images]

#         # 2) Turn messages into a single text string with the model’s chat template
#         #    (this adds any necessary <|user|>, <|assistant|> tokens and the generation prompt)
#         chat_text = self.processor.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True
#         )

#         # 3) Extract visual inputs (pixel buffers, bboxes, etc.)
#         image_inputs, video_inputs = process_vision_info(messages)  # returns two things; video_inputs will be None here :contentReference[oaicite:0]{index=0}

#         # 4) Tokenize the chat text
#         text_inputs = self.processor(
#             chat_text,
#             return_tensors="pt",
#             add_special_tokens=False  # tokens already handled by apply_chat_template
#         )

#         # 5) Merge text + images into one input dict
#         #    (drop video_inputs since you’re only doing stills)
#         inputs = {**text_inputs, **(image_inputs or {})}
#         inputs = {k: v.to(self.device) for k, v in inputs.items()}

#         # 6) Generate
#         defaults = dict(max_new_tokens=256, do_sample=False)
#         outputs = self.model.generate(**inputs, **{**defaults, **generate_kwargs})

#         # 7) Decode & strip off the echoed prompt
#         full = self.processor.decode(outputs[0], skip_special_tokens=True)
#         return full[len(prompt):].strip()

In [5]:

class LocalVisionLLM:
    def __init__(self, model_id: str, device_map: str = 'auto', torch_dtype="auto"):
        # 1) Load processor
        self.processor = AutoProcessor.from_pretrained(
            model_id,
            use_auth_token=True,
            trust_remote_code=True
        )
        # 2) Load model
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True,
            use_auth_token=True
        )
        # track actual device
        self.device = next(self.model.parameters()).device

    def __call__(self, images: list[Image.Image], prompt: str, max_new_tokens: int = 256, **gen_kwargs) -> str:
        # Build a single “user” message that contains both the images and the text
        messages = [
            {
                "role": "user",
                "content": [
                    *[
                        {"type": "image", "image": img}
                        for img in images
                    ],
                    {"type": "text", "text": prompt}
                ],
            }
        ]

        # 1) Format chat
        chat_text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # 2) Extract vision inputs
        image_inputs, video_inputs = process_vision_info(messages)

        # 3) Prepare model inputs (batched)
        model_inputs = self.processor(
            text=[chat_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        model_inputs = {k: v.to(self.device) for k, v in model_inputs.items()}

        # 4) Generate
        gen_kwargs = dict(max_new_tokens=max_new_tokens, **gen_kwargs)
        generated_ids = self.model.generate(**model_inputs, **gen_kwargs)

        # 5) Trim off the prompt tokens
        #    generated_ids is shape [batch, seq_len]; inputs.input_ids is [batch, seq_len_in]
        input_ids = model_inputs["input_ids"]
        trimmed = [
            out_ids[input_ids.shape[1]:]
            for out_ids in generated_ids
        ]

        # 6) Decode
        output_texts = self.processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        # single-example batch, so take [0]
        return output_texts[0].strip()

In [6]:
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
vlm = LocalVisionLLM(model_id)

/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/processing_auto.py:255: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/modeling_utils.py:4191: Fut

KeyboardInterrupt: 

In [ ]:
task = "same_second"
legend = False

if legend:
    sfx = "_wlegend"
else:
    sfx = ""
# task = "same_second"
# task = "same_first_global"
# csv_path = Path("groups_diff_first_results.csv")
# csv_path = Path("groups_same_second_results.csv")
csv_path = Path("groups_{}_results.csv".format(task))
data_dir = Path("./randomized_options/examples_{}{}".format(task, sfx))
df = pd.read_csv(csv_path)

In [ ]:

# def build_prompt(options: list[str]) -> str:
#     text = (
#         "I am showing you 5 apartment floorplan images.\n"
#         "4 share the same layout and 1 is different.\n\n"
#         "Please look carefully at spatial relationships, room types, and sizes.\n"
#         "Which example is the different one, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different example:** <ID>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#         "Examples by number:\n"
#     )
#     for idx, pid in enumerate(options, start=1):
#         text += f"- Example {idx}: ID {pid}\n"
#     return text

def build_prompt(options: list[str]) -> str:
    text = (
        "I am showing you five apartment floorplans, labeled A through E.\n"
        "One of these plans has a different underlying floorplan pattern, while the other four share the same pattern.\n\n"
        "There is a color legend indicating room-type colors.\n"
        "Please examine all five floorplans in terms of spatial relationships, room types, and relative sizes.\n\n"
        "Question: Which floorplan (A, B, C, D, or E) has a different underlying pattern, and why?\n\n"
        "Please structure your response exactly as follows:\n"
        "1. **Different floorplan:** <A/B/C/D/E>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )
    return text


def load_annotation(pid: str) -> str:
    """
    Load the human-annotated textual description for a given floorplan ID.
    """
    txt_path = ANNOT_DIR / f"{pid}.txt"
    try:
        return txt_path.read_text(encoding='utf-8').strip()
    except FileNotFoundError:
        return "<No annotation available>"

def build_2modal_prompt(options: list[str]) -> str:
    letters = ['A', 'B', 'C', 'D', 'E']

    # Intro and setup
    text = (
        "I am presenting five apartment floorplans, labeled A through E.\n"
        "One plan has a different underlying floorplan pattern; the other four share the same design.\n\n"
        "Below are all annotated descriptions, each correspond to an image labeled A–E. Please read them carefully.\n\n"
        "Descriptions:\n"
    )

    # List all descriptions together
    for label, pid in zip(letters, options):
        annotation = load_annotation(pid)
        text += f"  {label}. {annotation}\n"

    text += (
        "\nHere is the combined image showing floorplans A through E side by side.\n"
        "Examine all five floorplans in terms of spatial layout, room types, and relative sizes.\n"
        "Question: Which floorplan (A, B, C, D, or E) has a different underlying pattern, and why?\n\n"
        "Please structure your response exactly as follows:\n"
        "1. **Different floorplan:** <A/B/C/D/E>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )

    return text

# def build_2modal_prompt(options: list[str]) -> str:
#     """
#     Creates a prompt for a consolidated 2-modality input: five floorplans labeled A–E, each with a human description.
#     """
#     letters = ['A', 'B', 'C', 'D', 'E']
#     text = (
#         "I am presenting you five apartment floorplans, labeled A through E.\n"
#         "One of these plans has a different underlying floorplan pattern, while the other four share the same pattern.\n\n"
#         "Each plan has a human-annotated description as shown below.\n\n"
#         "Descriptions:\n"
#     )

#     for label, pid in zip(letters, options):
#         annotation = load_annotation(pid)
#         text += f"- {label}: {annotation}\n"

#     text += (
#         "Here I am also showing you an image with all five floorplans, labeled A through E, corresponding to the descriptions A through E.\n\n"
#         "Please examine spatial relationships, room types, and sizes based on the descriptions and images.\n\n"
#         "Which of A, B, C, D, or E has a different underlying floorplan pattern from others, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different plan:** <A/B/C/D/E>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#     )
#     return text

In [ ]:
results = []

reoriented = False

multi = True

if legend:
    f_sfx = "wlegend"
else:
    f_sfx = "nolegend"

if reoriented:
    f_suffix = "reoriented"
else:
    f_suffix = "original"

if reoriented:
    suffix = ""
else:
    suffix = "_2"

for i, row in df.iterrows():
    # Each row points to one consolidated image file, e.g. case_01.png
    case_id = i + 1  # adjust column name as needed
    options = row['options']

    img_path = data_dir / f"example_{case_id}{suffix}.png"

    # Open the single consolidated image
    consolidated_img = Image.open(img_path).convert("RGB")

    # Build prompt
    label_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}  # if needed

    if multi:
        prompt = build_2modal_prompt(options)
    else:
        prompt = build_prompt(label_map)

    # Run the VLM model on the single image
    answer = vlm([consolidated_img], prompt, do_sample=False)

    # Parse out the letter from the first line
    first_line = answer.splitlines()[0]
    predicted_label = first_line.split(":", 1)[1].strip()

    results.append({
        'base_cluster':  row['base_cluster'],
        'other_cluster': row['other_cluster'],
        'options':       options,
        'outlier_id':    row['outlier_id'],
        'predicted_id':  predicted_label,
        'raw_response':  answer
    })
# save
df_out = pd.DataFrame(results)

if not multi:
    out_path = Path("outputs/vlm_qwen2.5-vl_img_{}_{}_{}.csv".format(task, f_suffix, f_sfx))
else:
    out_path = Path("outputs/vlm_qwen2.5-vl_multi_{}_{}_{}.csv".format(task, f_suffix, f_sfx))
out_path.parent.mkdir(exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Wrote {len(df_out)} results to {out_path}")

PosixPath('../../data/floorplan_image')

In [ ]:
tasks = ['same_second', 'diff_first']
legend_opts = [True, False]
orient_opts = [True, False]
multi_opts = [True, False]

# map label->int if needed
label_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}

for task in tasks:
    for legend in legend_opts:
        for reoriented in orient_opts:
            for multi in multi_opts:
                # Determine suffixes and paths
                sfx = '_wlegend' if legend else ''
                f_sfx = 'wlegend' if legend else 'nolegend'
                f_suffix = 'reoriented' if reoriented else 'original'
                suffix = '' if reoriented else '_2'

                csv_path = Path(f"groups_{task}_results.csv")
                data_dir = Path(f"./randomized_options/examples_{task}{sfx}")

                # Read input data
                df = pd.read_csv(csv_path)
                results = []

                for i, row in df.iterrows():
                    case_id = i + 1
                    options = row['options']
                    img_path = data_dir / f"example_{case_id}{suffix}.png"

                    # Load image
                    consolidated_img = Image.open(img_path).convert("RGB")

                    # Build prompt
                    if multi:
                        prompt = build_2modal_prompt(options)
                    else:
                        prompt = build_prompt(options)

                    # Query VLM
                    answer = vlm([consolidated_img], prompt, do_sample=False)

                    # Parse prediction
                    first_line = answer.splitlines()[0]
                    predicted_label = first_line.split(":", 1)[1].strip()

                    results.append({
                        'base_cluster':  row['base_cluster'],
                        'other_cluster': row['other_cluster'],
                        'options':       options,
                        'outlier_id':    row['outlier_id'],
                        'predicted_id':  predicted_label,
                        'raw_response':  answer
                    })

                # Save outputs
                df_out = pd.DataFrame(results)
                mode = 'vlm_qwen2.5-vl'
                type_str = 'multi' if multi else 'img'
                out_dir = Path('outputs') / mode
                out_dir.mkdir(parents=True, exist_ok=True)
                out_name = f"{mode}_{type_str}_{task}_{f_suffix}_{f_sfx}.csv"
                out_path = out_dir / out_name
                df_out.to_csv(out_path, index=False)
                print(f"Wrote {len(df_out)} results to {out_path}")
